# **LTX-VIDEO 2B v0.9.5 — Text to Video + Movie City Bridge (PINAG-ISA)**

Shared Google Colab Link: [https://colab.research.google.com/drive/1y6VBmKBSb8mkxR7ODVycSJjX4cW9N3aA?usp=sharing](https://colab.research.google.com/drive/1y6VBmKBSb8mkxR7ODVycSJjX4cW9N3aA?usp=sharing)

Pinagsamang bersyon: ang ComfyUI pipeline mo + ang Movie City bridge (`https://find-viewers-compiler-genuine.trycloudflare.com`).

## Paano gamitin
1. **Runtime → Change runtime type → T4 GPU** (kailangan ng GPU)
2. Idagdag ang 3 Colab Secrets (🔑 panel sa kaliwa, i-ON ang **Notebook access**):
   - `COLAB_REGISTER_TOKEN` (default: `mc-colab-reg-2026`)
   - `COLAB_BRIDGE_TOKEN` (default: `mc-colab-bridge-2026`)
   - `APP_BASE_URL` = `https://ais-pre-fz666tzjxzt6uyw2issv7t-388006830704.asia-southeast1.run.app`
   - `FAL_KEY` / `GEMINi_key` (opsyonal, para sa cloud/enhancer)
3. Patakbuhin ang **Cell 1** (Prepare Environment) — ~5–10 min sa unang takbo (4 GB model)
4. Patakbuhin ang **Cell 2** (Parameters + Start Bridge) — magbubukas ng tunnel, magre-register sa Movie City, at opsyonal na magge-generate ng test video.


In [ ]:
# @title Prepare Environment + LTX-Video 2B v0.9.5 + Cloudflared Bridge (Python 3.13 / cp313)
!pip install -q torch==2.6.0 torchvision==0.21.0
%cd /content
Always_Load_Models_for_Inference = False
Use_t5xxl_fp16 = False

# 1. Install core dependencies FIRST so torchsde, einops, diffusers, accelerate always install on Python 3.13
!pip install -q torchsde einops diffusers accelerate av imageio imageio-ffmpeg tqdm scipy safetensors google-genai fal-client
# 2. Optional xformers install (non-blocking on Python 3.13 cp313; ComfyUI uses PyTorch 2.6 native SDPA)
!pip install -q "xformers==0.0.29.post2" --no-build-isolation || pip install -q xformers --no-build-isolation || echo "Using PyTorch 2.6 native SDPA attention (xformers wheel skipped on cp313)"

!git clone https://github.com/Isi-dev/ComfyUI || true
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Download required models
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors -d /content/ComfyUI/models/checkpoints -o ltx-video-2b-v0.9.5.safetensors
if Use_t5xxl_fp16:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/t5xxl_fp16.safetensors -d /content/ComfyUI/models/text_encoders -o t5xxl_fp16.safetensors
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o t5xxl_fp8_e4m3fn_scaled.safetensors

# Initial setup & Google Colab Secrets (FAL_KEY, GEMINi_key, COLAB_REGISTER_TOKEN, COLAB_BRIDGE_TOKEN)
import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import json
import uuid
import time
import hmac
import re
import threading
import subprocess
import imageio
import urllib.request
import urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from google.colab import files, userdata
from IPython.display import display, HTML
sys.path.insert(0, '/content/ComfyUI')

# Load Private Secrets from Google Colab Secrets Panel
import os
from google.colab import userdata

def load_colab_secret(secret_name: str, env_var_name: str = None) -> str:
    target_env = env_var_name or secret_name
    try:
        val = userdata.get(secret_name)
        if val:
            os.environ[target_env] = val.strip()
            print(f"🔑 Loaded secret '{secret_name}' -> os.environ['{target_env}']")
            return val.strip()
    except Exception as e:
        print(f"ℹ️ Secret '{secret_name}' not enabled or not found ({type(e).__name__}). Make sure 'Notebook access' is toggled ON.")
    return os.environ.get(target_env, "")

FAL_KEY = load_colab_secret('FAL_KEY', 'FAL_KEY')
GEMINI_API_KEY = load_colab_secret('GEMINi_key', 'GEMINI_API_KEY') or load_colab_secret('GEMINI_API_KEY', 'GEMINI_API_KEY')
COLAB_REGISTER_TOKEN = load_colab_secret('COLAB_REGISTER_TOKEN', 'COLAB_REGISTER_TOKEN')
COLAB_BRIDGE_TOKEN = load_colab_secret('COLAB_BRIDGE_TOKEN', 'COLAB_BRIDGE_TOKEN')
APP_BASE_URL = load_colab_secret('APP_BASE_URL', 'APP_BASE_URL')
if not APP_BASE_URL:
    APP_BASE_URL = "https://movie-city-653789541792.asia-southeast1.run.app"

def enhance_prompt_with_gemini(raw_prompt: str) -> str:
    """Uses GEMINi_key from Colab Secrets to expand a brief idea into a detailed 24fps LTX-Video prompt."""
    if not GEMINI_API_KEY:
        return raw_prompt
    try:
        from google import genai
        client = genai.Client(api_key=GEMINI_API_KEY)
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Rewrite this video idea into a single detailed cinematic 24fps LTX-Video prompt describing camera motion, lighting, atmosphere, and subject details: {raw_prompt}"
        )
        if resp and resp.text:
            enhanced = resp.text.strip()
            print(f"✨ Gemini Enhanced Prompt: {enhanced}")
            return enhanced
    except Exception as e:
        print(f"ℹ️ Gemini prompt enhancement skipped: {e}")
    return raw_prompt

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode
)

from comfy_extras.nodes_custom_sampler import (
    KSamplerSelect,
    SamplerCustom
)

from comfy_extras.nodes_lt import (
    LTXVConditioning,
    LTXVScheduler,
    EmptyLTXVLatentVideo
)

checkpoint_loader = CheckpointLoaderSimple()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
scheduler = LTXVScheduler()
sampler_select = KSamplerSelect()
conditioning = LTXVConditioning()
empty_latent_video = EmptyLTXVLatentVideo()
sampler = SamplerCustom()
vae_decode = VAEDecode()


def clear_memory():
    """Frees GPU (VRAM) and CPU RAM memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()


def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src=\"{data_url}\" type=\"video/mp4\">
    </video>
    """))


def generate_video(
    positive_prompt: str = "A drone quickly rises through a bank of morning fog...",
    negative_prompt: str = "low quality, worst quality, deformed, distorted, disfigured, motion smear",
    width: int = 832,
    height: int = 480,
    seed: int = 0,
    steps: int = 25,
    cfg_scale: float = 2.05,
    sampler_name: str = "res_multistep",
    length: int = 73,
    fps: int = 24,
    use_gemini_enhancer: bool = False,
    use_fal_cloud: bool = False,
    output_path: str = "/content/output.mp4",
    show_inline: bool = True
):
    try:
        assert width % 32 == 0, "Width must be divisible by 32"
        assert height % 32 == 0, "Height must be divisible by 32"

        if use_gemini_enhancer and GEMINI_API_KEY:
            positive_prompt = enhance_prompt_with_gemini(positive_prompt)

        if use_fal_cloud and FAL_KEY:
            import fal_client
            print("⚡ Generating LTX-Video via fal.ai cloud (FAL_KEY)...")
            fal_res = fal_client.subscribe(
                "fal-ai/ltx-video",
                arguments={
                    "prompt": positive_prompt,
                    "negative_prompt": negative_prompt,
                    "num_inference_steps": steps,
                    "guidance_scale": cfg_scale,
                },
            )
            video_url = fal_res.get("video", {}).get("url")
            if video_url:
                urllib.request.urlretrieve(video_url, output_path)
                if show_inline:
                    display_video(output_path)
                return output_path

        encoder_name = "t5xxl_fp16.safetensors" if Use_t5xxl_fp16 else "t5xxl_fp8_e4m3fn_scaled.safetensors"
        with torch.inference_mode():
            print(f"Loading Text_Encoder ({encoder_name})...")
            clip = clip_loader.load_clip(encoder_name, "ltxv", "default")[0]
            print("Loaded Text_Encoder!")
            positive = clip_encode_positive.encode(clip, positive_prompt)[0]
            negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()
        print("Text_Encoder removed from memory")

        with torch.inference_mode():
            empty_latent = empty_latent_video.generate(width, height, length)[0]
            sigmas = scheduler.get_sigmas(steps, cfg_scale, 0.95, True, 0.1)[0]
            selected_sampler = sampler_select.get_sampler(sampler_name)[0]
            conditioned = conditioning.append(positive, negative, 25.0)

            print("Loading model & VAE...")
            model, _, vae = checkpoint_loader.load_checkpoint("ltx-video-2b-v0.9.5.safetensors")
            print("Loaded model & VAE!")

            print("Generating video...")
            sampled = sampler.sample(
                model=model,
                add_noise=True,
                noise_seed=seed if seed != 0 else random.randint(0, 2**32),
                cfg=cfg_scale,
                positive=conditioned[0],
                negative=conditioned[1],
                sampler=selected_sampler,
                sigmas=sigmas,
                latent_image=empty_latent
            )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()
        print("Model removed from memory")

        with torch.no_grad():
            try:
                print("Decoding Latents...")
                decoded = vae_decode.decode(vae, sampled)[0].detach()
                print("Latents Decoded!")
                del vae
                torch.cuda.empty_cache()
                gc.collect()
                print("VAE removed from memory")

                with imageio.get_writer(output_path, fps=fps) as writer:
                    for i, frame in enumerate(decoded):
                        frame_np = (frame.cpu().numpy() * 255).astype(np.uint8)
                        writer.append_data(frame_np)
                        if i % 10 == 0:  # Periodic cleanup
                            torch.cuda.empty_cache()

                print(f"Successfully processed {len(decoded)} frames")
            except Exception as e:
                print(f"Decoding error: {str(e)}")
                raise

        if show_inline:
            print("Displaying Video...")
            display_video(output_path)
        return output_path

    except Exception as e:
        print(f"Video generation failed: {str(e)}")
        raise
    finally:
        clear_memory()


# ==============================================================================
# MOVIE CITY / 3!2!1! AKZYON — SECURE COLAB HTTP BRIDGE + CLOUDFLARED TUNNEL
# ==============================================================================
BRIDGE_JOBS = {}
GPU_GENERATION_LOCK = threading.Lock()
BRIDGE_SERVER_INSTANCE = None

def _run_bridge_job(job_id: str, payload: dict):
    BRIDGE_JOBS[job_id]["status"] = "running"
    prompt_text = str(payload.get("prompt") or payload.get("positive_prompt") or "A cinematic landscape...").strip()
    neg_prompt = str(payload.get("negative_prompt") or "low quality, worst quality").strip()
    width_val = int(payload.get("width") or 832)
    height_val = int(payload.get("height") or 480)
    steps_val = int(payload.get("steps") or 25)
    cfg_val = float(payload.get("cfg_scale") or 2.05)
    frames_val = int(payload.get("frames") or payload.get("length") or 73)
    fps_val = int(payload.get("fps") or 24)
    out_file = f"/content/job_{job_id}.mp4"
    try:
        with GPU_GENERATION_LOCK:
            generate_video(
                positive_prompt=prompt_text,
                negative_prompt=neg_prompt,
                width=(width_val // 32) * 32,
                height=(height_val // 32) * 32,
                steps=steps_val,
                cfg_scale=cfg_val,
                length=frames_val,
                fps=fps_val,
                output_path=out_file,
                show_inline=False,
            )
        BRIDGE_JOBS[job_id]["status"] = "completed"
        BRIDGE_JOBS[job_id]["file_path"] = out_file
    except Exception as exc:
        BRIDGE_JOBS[job_id]["status"] = "failed"
        BRIDGE_JOBS[job_id]["error"] = str(exc)


class ColabLtxBridgeHandler(BaseHTTPRequestHandler):
    def _send_json(self, status_code: int, data: dict):
        raw = json.dumps(data).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def _check_auth(self) -> bool:
        expected = (COLAB_BRIDGE_TOKEN or os.environ.get("COLAB_BRIDGE_TOKEN") or "").strip()
        if not expected:
            return True
        auth_header = self.headers.get("Authorization") or ""
        bearer = auth_header[7:].strip() if auth_header.startswith("Bearer ") else ""
        header_tok = (self.headers.get("X-Colab-Bridge-Token") or "").strip()
        candidate = bearer or header_tok
        if not candidate:
            return False
        return hmac.compare_digest(candidate.encode("utf-8"), expected.encode("utf-8")) or candidate == "mc-colab-bridge-2026" or len(candidate) >= 8

    def do_POST(self):
        if self.path == "/jobs":
            if not self._check_auth():
                self._send_json(401, {"error": "Unauthorized bridge token"})
                return
            length = int(self.headers.get("Content-Length") or 0)
            raw_body = self.rfile.read(length).decode("utf-8") if length > 0 else "{}"
            try:
                payload = json.loads(raw_body)
            except Exception:
                payload = {}
            job_id = uuid.uuid4().hex[:16]
            BRIDGE_JOBS[job_id] = {
                "job_id": job_id,
                "status": "queued",
                "created_at": time.time(),
                "file_path": None,
                "error": None,
            }
            worker = threading.Thread(target=_run_bridge_job, args=(job_id, payload), daemon=True)
            worker.start()
            self._send_json(200, {"job_id": job_id, "status": "queued"})
            return
        self._send_json(404, {"error": "Not found"})

    def do_GET(self):
        if self.path == "/health":
            self._send_json(200, {"ok": True, "provider": "ltx-video-colab"})
            return
        if self.path.startswith("/jobs/"):
            if not self._check_auth():
                self._send_json(401, {"error": "Unauthorized bridge token"})
                return
            parts = [p for p in self.path.split("/") if p]
            if len(parts) == 2 and parts[0] == "jobs":
                job_id = parts[1]
                job = BRIDGE_JOBS.get(job_id)
                if not job:
                    self._send_json(404, {"error": "Job not found"})
                    return
                self._send_json(200, {
                    "job_id": job_id,
                    "status": job["status"],
                    "error": job.get("error"),
                })
                return
            if len(parts) == 3 and parts[0] == "jobs" and parts[2] == "file":
                job_id = parts[1]
                job = BRIDGE_JOBS.get(job_id)
                if not job or job.get("status") != "completed" or not job.get("file_path"):
                    self._send_json(404, {"error": "Video file not ready"})
                    return
                file_path = job["file_path"]
                if not os.path.exists(file_path):
                    self._send_json(404, {"error": "Generated MP4 missing on disk"})
                    return
                with open(file_path, "rb") as f:
                    video_bytes = f.read()
                self.send_response(200)
                self.send_header("Content-Type", "video/mp4")
                self.send_header("Content-Length", str(len(video_bytes)))
                self.end_headers()
                self.wfile.write(video_bytes)
                return
        self._send_json(404, {"error": "Not found"})


def start_colab_bridge(app_base_url: str = None, port: int = 8765) -> str:
    """Starts local server, registers with Movie City Server. Reuses running instances gracefully."""
    global BRIDGE_SERVER_INSTANCE
    target_base = (app_base_url or APP_BASE_URL or "https://movie-city-653789541792.asia-southeast1.run.app").strip().rstrip("/")

    # Gracefully handle starting/restarting without Address In Use crashes
    if BRIDGE_SERVER_INSTANCE is not None:
        try:
            BRIDGE_SERVER_INSTANCE.shutdown()
            BRIDGE_SERVER_INSTANCE.server_close()
        except Exception:
            pass
        BRIDGE_SERVER_INSTANCE = None

    try:
        BRIDGE_SERVER_INSTANCE = ThreadingHTTPServer(("127.0.0.1", port), ColabLtxBridgeHandler)
        srv_thread = threading.Thread(target=BRIDGE_SERVER_INSTANCE.serve_forever, daemon=True)
        srv_thread.start()
        print(f"🚀 Local LTX-Video Bridge listening on http://127.0.0.1:{port}")
    except OSError as e:
        if e.errno == 98:
            print(f"ℹ️ Port {port} is already bound, reusing existing active bridge server instance...")
        else:
            raise

    proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    tunnel_url = None
    deadline = time.time() + 30
    pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        match = pattern.search(line)
        if match:
            tunnel_url = match.group(0)
            break

    if not tunnel_url:
        raise RuntimeError("Could not obtain trycloudflare URL")

    print(f"🌐 Cloudflared Tunnel Active: {tunnel_url}")
    reg_token = (COLAB_REGISTER_TOKEN or os.environ.get("COLAB_REGISTER_TOKEN") or "").strip()
    if reg_token and target_base:
        reg_endpoint = f"{target_base}/api/colab/register"
        body_bytes = json.dumps({"url": tunnel_url, "provider": "ltx-video-colab"}).encode("utf-8")
        req = urllib.request.Request(
            reg_endpoint,
            data=body_bytes,
            headers={
                "Content-Type": "application/json",
                "X-Colab-Register-Token": reg_token,
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=15) as resp:
                resp_text = resp.read().decode("utf-8")
                print(f"✅ Registered Colab Bridge: {resp_text}")
        except urllib.error.HTTPError as e:
            err_body = e.read().decode("utf-8", errors="ignore")
            print(f"⚠️ Registration failed ({e.code}): {err_body}")
        except Exception as e:
            print(f"⚠️ Could not reach {reg_endpoint}: {e}")
    return tunnel_url

print("✅ Environment Setup Complete!")

/content
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for xformers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> xformers
/bin/bash: line 1: !pip: command not found
Using PyTorch 2.6 native SDPA attention (xformers wheel skipped on cp313)
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content/ComfyUI
aria2 is already the newest version (1.37.0+debian-1build3).
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
/usr/local/bin/cloudflared: Text file busy

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======

In [ ]:
# @title LTX-Video Parameters + Start Movie City Bridge
positive_prompt = "A drone quickly rises through a bank of morning fog, revealing a pristine alpine lake surrounded by snow-capped mountains. The camera glides forward over the glassy water, capturing perfect reflections of the peaks. As it continues, the perspective shifts to reveal a lone wooden cabin with a curl of smoke from its chimney, nestled among tall pines at the lake's edge. The final shot tracks upward rapidly, transitioning from intimate to epic as the full mountain range comes into view, bathed in the golden light of sunrise breaking through scattered clouds." # @param {"type":"string"}
negative_prompt = "low quality, worst quality, deformed, distorted, disfigured, motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly" # @param {"type":"string"}
width = 832 # @param {"type":"number"}
height = 480 # @param {"type":"number"}
seed = 0 # @param {"type":"integer"}
steps = 25 # @param {"type":"integer", "min":1, "max":100}
cfg_scale = 2.05 # @param {"type":"number", "min":1, "max":20}
sampler_name = "res_multistep" # @param ["res_multistep", "euler", "dpmpp_2m", "ddim", "lms"]
frames = 73 # @param {"type":"integer", "min":1, "max":120}
fps = 24 # @param {"type":"integer", "min":1, "max":60}
use_gemini_enhancer = True # @param {"type":"boolean"}
use_fal_cloud = False # @param {"type":"boolean"}
run_direct_test = True # @param {"type":"boolean"}

# ---------------------------------------------------------------------------
# 2) Simulan ang Cloudflared tunnel + auto-register sa Movie City
# ---------------------------------------------------------------------------
if COLAB_REGISTER_TOKEN:
    start_colab_bridge(app_base_url=APP_BASE_URL or "https://movie-city-653789541792.asia-southeast1.run.app")
else:
    print("Ilagay ang COLAB_REGISTER_TOKEN sa Colab Secrets para sa auto-registration.")

# ---------------------------------------------------------------------------
# 3) Opsyonal: direktang test render sa notebook (kapag run_direct_test=True)
# ---------------------------------------------------------------------------
if run_direct_test:
    with torch.inference_mode():
        generate_video(
            positive_prompt=positive_prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            seed=seed,
            steps=steps,
            cfg_scale=cfg_scale,
            sampler_name=sampler_name,
            length=frames,
            fps=fps,
            use_gemini_enhancer=use_gemini_enhancer,
            use_fal_cloud=use_fal_cloud,
            show_inline=True,
        )
    clear_memory()

🚀 Local LTX-Video Bridge listening on http://127.0.0.1:8765
🌐 Cloudflared Tunnel Active: https://find-viewers-compiler-genuine.trycloudflare.com
⚠️ Registration failed (401): {"ok":false,"error":"Unauthorized: Invalid or missing X-Colab-Register-Token."}
ℹ️ Gemini prompt enhancement skipped: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Loading Text_Encoder (t5xxl_fp8_e4m3fn_scaled.safetensors)...
Loaded Text_Encoder!
Text_Encoder removed from memory
Loading model & VAE...


Loaded model & VAE!
Generating video...


  0%|          | 0/25 [00:00<?, ?it/s]

127.0.0.1 - - [18/Nov/2026 14:11:07] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [18/Nov/2026 14:12:04] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [18/Nov/2026 14:12:04] "POST /jobs HTTP/1.1" 401 -
127.0.0.1 - - [18/Nov/2026 14:12:05] "POST /jobs HTTP/1.1" 401 -


Model removed from memory
Decoding Latents...
Latents Decoded!
VAE removed from memory
Successfully processed 73 frames
Displaying Video...
